In [2]:
# Tecnología
import json
import calendar
import pandas as pd
from sparky_bc import Sparky
import datetime as dt
from dateutil.relativedelta import relativedelta

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'

# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
# sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp", spark_submit="spark3-submit")
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-06-14 10:06:52 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/ADQUIRENCIA_FERIA_EVA/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



# Actualización

In [ ]:
### LEER ATENTAMENTE ###
# Las tablas a actualizar histórico o crear histórico son: `proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist` y `proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist`
# Modificar los valores de las siguientes variables cada vez que se creen históricos o se realicen actualizaciones.

# ¿Va a actualizar o construir histórico?
actualizar = False # MODIFICAR. HISTORICO ENTONCES USAR False. ACTUALIZAR ENTONCES USAR True.

# Crear tabla con transacciones adquirencia
# Si se va a actualizar garantizar que la fecha_inicial_trxs sea siempre el primer día del año corrido
# Si se va a crear histórico, garantizar que la fecha_inicial_trxs sea el primer día del año del histórico a crear.
# La fecha_final_trxs siempre debe ser esa fecha que garantice la completitud de las transacciones del último mes. Verificarlo con el código de la sesión Análisis ingestión compras tabla transaccional
# Por lo general para tener la completitud de transacciones de un mes, se requiere las ingestion de transacciones del mes siguiente.
fecha_inicial_trxs = '2022-01-01' # MODIFICAR.
fecha_final_trxs = '2026-06-09' # MODIFICAR.

# Almacenar histórico o actualización de métricas en tablas `proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist` y `proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist`
# Si va a actualiar verifique que las tablas existen
# y verifique el rango del histórico almacenado para determinar desde donde iniciar la actualización.
# Rango histórico almacenado en las tablas: 202201 hasta 202605. MODIFICAR.
fecha_inicial_metricas = '2022-01-01' # MODIFICAR
fecha_final_metricas = '2026-05-31' # MODIFICAR

# Leer archivo con el primer día de ingestión de cada mes de la tabla resultados_vspc_medios_de_pago.gsap_m_comercios
# Este archivo lo generá el archivo vinculacion_adquirencia.ipynb
### OPORTUNIDAD DE MEJORA: automatizar la generación de este archivo para no tener que actualizarlo manualmente. #####
# Se tienen datos desde 202201 hasta 202606 # MODIFICAR
df_primer_dia_ing_gsap_m_comercios = pd.read_excel('main_data/df_primer_dia_ing_vinc.xlsx')
df_primer_dia_ing_gsap_m_comercios.columns = ['year_sgte_mes', 'month_sgte_mes', 'primer_dia_ing_gsap_m_comercios', 'frec']

# El ciclo de vida de la zona proceso_vdm es de 30 días
# La última vez que se borró y puso de nuevo las tablas 
# fue el 2026-06-14 # MODIFICAR.

## Uso o activo

Un comercio usa adquirencia en un mes específico o mes de análisis cuando:

- Métrica Normal: tienen al menos 1 trx aporbada en ese mes.

- Métrica 5x: tiene al menos 1 trx aprobada en el transucurso de un año, incluyendo el mes de análisis


### Análisis ingestión compras tabla transaccional

In [9]:
# Se deben esperar dos días para que se ingeste la información completa de las transacciones de un día en particular
# Para obtener la información de las compras del día jueves y viernes se debe espear hasta la sgte semana
sql = """
WITH outcome1 AS
  (SELECT YEAR,
          MONTH,
          DAY,
          left(cast(f_trx as string), 10) as f_trx,
          count(*) AS num_compras
   FROM resultados_vspc_medios_de_pago.gsap_m_transaccional
   WHERE YEAR IN (2026)
     AND MONTH IN (5, 6)
     AND DAY BETWEEN 1 AND 31
     and tipo_trx = "Purchase"
   GROUP BY 1,
            2,
            3,
            4
   ORDER BY f_trx DESC, YEAR DESC, MONTH DESC, DAY DESC),
     outcome2 AS
  (SELECT *,
          sum(num_compras) OVER (PARTITION BY f_trx) AS total_compras,
                                sum(num_compras) OVER (PARTITION BY f_trx
                                                       ORDER BY YEAR,
                                                                MONTH,
                                                                DAY ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_compras_cumsum
   FROM outcome1
   ORDER BY f_trx DESC, YEAR DESC, MONTH DESC, DAY DESC)
SELECT *,
       row_number() OVER (PARTITION BY f_trx
                          ORDER BY YEAR,
                                   MONTH,
                                   DAY) AS num_ing,
                         round(num_compras /total_compras, 4) AS prop,
                         round(num_compras_cumsum / total_compras, 4) AS prop_cumsum
FROM outcome2
ORDER BY f_trx DESC,
         YEAR DESC, MONTH DESC, DAY DESC;
"""
df_prueba = helper.obtener_dataframe(sql)

-----------------------------------------------------------------------------------------------
  i    tipo                     nombre                     estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------------------
 4/4 DATAFRAME                                           descargando   07:42:55 PM             

2026-06-10 19:43:00 - [INFO] - 877 filas, 10 columnas, 00:04.8 consultando, 00:00.2 descargando, 00:00.0 convirtiendo


 4/4 DATAFRAME                                            finalizado   07:42:55 PM     00:05.1 
-----------------------------------------------------------------------------------------------


In [13]:
df_prueba[365:415]

,year,month,day,f_trx,num_compras,total_compras,num_compras_cumsum,num_ing,prop,prop_cumsum
365,2026,5,11,2026-05-02,1,2529803,2529783,5,0.0000,1.0000
366,2026,5,7,2026-05-02,3,2529803,2529782,4,0.0000,1.0000
367,2026,5,6,2026-05-02,1,2529803,2529779,3,0.0000,1.0000
368,2026,5,5,2026-05-02,243,2529803,2529778,2,0.0001,1.0000
369,2026,5,4,2026-05-02,2529535,2529803,2529535,1,0.9999,0.9999
370,2026,6,9,2026-05-01,2,2509603,2509603,11,0.0000,1.0000
371,2026,6,3,2026-05-01,21,2509603,2509601,10,0.0000,1.0000
372,2026,5,29,2026-05-01,1,2509603,2509580,9,0.0000,1.0000
373,2026,5,26,2026-05-01,23,2509603,2509579,8,0.0000,1.0000
374,2026,5,21,2026-05-01,8,2509603,2509556,7,0.0000,1.0000


## Construcción histórico transacciones

In [ ]:
# Construcción histórico trxs
fecha_inicial_ts = pd.to_datetime(fecha_inicial_trxs)
fecha_final_ts = pd.to_datetime(fecha_final_trxs)
fechas = pd.date_range(start=fecha_inicial_ts, end=fecha_final_ts, freq='D')

# # Ir por las particiones del siguiente mes que contendrá las compras de los últimos días del reango de tiempo deseado
# pri_dia_part = fechas[-1] + relativedelta(days=1)
# pri_dia_part = pri_dia_part.date().isoformat() # Primer día de partición para buscar compras del primer día.
# ult_dia_part = fechas[-1] + relativedelta(days=10)
# ult_dia_part = ult_dia_part.date().isoformat()
# fechas_faltantes = pd.date_range(start=pri_dia_part, end=ult_dia_part, freq='D')

# # siguiente_dias = dt.datetime(siguientes_dias.year, siguientes_dias.month, calendar.monthrange(siguientes_dias.year, siguientes_dias.month)[1])
# fechas = fechas.append(fechas_faltantes)

# df_config
df_config = pd.DataFrame({'fechas': fechas})
df_config['year'] = df_config['fechas'].dt.year
df_config['month'] = df_config['fechas'].dt.month
df_config['day'] = df_config['fechas'].dt.day
df_config

,fechas,year,month,day
0,2026-02-12,2026,2,12
1,2026-02-13,2026,2,13
2,2026-02-14,2026,2,14
3,2026-02-15,2026,2,15
4,2026-02-16,2026,2,16
...,...,...,...,...
113,2026-06-05,2026,6,5
114,2026-06-06,2026,6,6
115,2026-06-07,2026,6,7
116,2026-06-08,2026,6,8


In [ ]:
# Crear tabla que almacenará la información
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_trxs;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_adquirencia_trxs  (
                cod_unico VARCHAR,
                f_trx STRING,
                num_trxs BIGINT,
                mnt_total_trxs DECIMAL(38,2),
                DIA INT
                )
            PARTITIONED BY 
            (
            YEAR INT,
            MES INT
            )
STORED AS PARQUET
TBLPROPERTIES ('transactional' = 'false');
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_trxs;"""
helper.ejecutar_consulta(sql_compute)

2026-06-10 21:25:34 - [INFO] - Transcurrido: 4368, Tiempo de Refresco = 1000


---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------------
 379/379      DROP         proceso_vdm.mdo_adquirencia_trxs   finalizado   09:25:35 PM     00:00.8 
---------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------------
 380/380    CREATE         proceso_vdm.mdo_adquirencia_trxs   finalizado   09:25:36 PM     00:00.6 
---------------------------------------------------------------------------------------------------


In [ ]:
# Iterar para obtener las trxs por cliente
for row in df_config.itertuples():
    print('#' * 50)
    print('')
    print('Exrayendo datos de las particiones: ', str(row.year), '-', str(row.month), '-', str(row.day))
    print('')
    print('Obteniendo transacciones adquirencia de los comercios')
    print('')

    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_trxs_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)

    sql = """
    CREATE TABLE proceso.mdo_adquirencia_trxs_temp STORED AS PARQUET AS
    SELECT cod_unico,
       to_date(f_trx) as f_trx,
       count(*) AS num_trxs,
       sum(mnt_total_trx) AS mnt_total_trxs,
        """ + str(row.day) + """ AS DIA,
        """ + str(row.year) + """ AS YEAR,
        """ + str(row.month) + """ AS MES
    FROM resultados_vspc_medios_de_pago.gsap_m_transaccional
    WHERE YEAR = """ + str(row.year) + """
     AND MONTH = """ + str(row.month) + """
     AND DAY = """ + str(row.day) + """
     AND LOWER(TRIM(tipo_trx)) = "purchase"
     AND LOWER(TRIM(estado_trx)) = "cleared"
    GROUP BY 1,
            2;"""
    helper.ejecutar_consulta(sql)

    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_trxs_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando transacciones adquirencia de los comercios')
    print('')

    sql = """
    INSERT INTO proceso.mdo_adquirencia_trxs PARTITION (YEAR = """ + str(row.year) + """, MES = """ + str(row.month) + """)
    SELECT cod_unico,
           f_trx,
           num_trxs,
           mnt_total_trxs,
           DIA
    FROM proceso.mdo_adquirencia_trxs_temp;"""
    helper.ejecutar_consulta(sql)
    print('')

2026-06-10 23:30:19 - [INFO] - Transcurrido: 1781152220, Tiempo de Refresco = 1000


##################################################

Exrayendo datos de las particiones:  2026 - 2 - 12

Obteniendo transacciones adquirencia de los comercios

-----------------------------------------------------------------------------------
  i  tipo              nombre                  estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------
 1/1 DROP proceso.mdo_adquirencia_trxs_temp   finalizado   11:30:21 PM     00:00.9 
-----------------------------------------------------------------------------------
-------------------------------------------------------------------------------------
  i   tipo               nombre                  estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------
 2/2 CREATE proceso.mdo_adquirencia_trxs_temp   finalizado   11:30:22 PM     00:01.5 
-----------------------------------------------------------------------------

In [ ]:
# # Verficar cantidad de registros desde la tabla fuente
# sql = """
# with outcome as (
# SELECT cod_unico,
#        to_date(f_trx) as f_trx,
#        count(*) AS num_trxs,
#        sum(mnt_total_trx) AS mnt_total_trxs
#     FROM resultados_vspc_medios_de_pago.gsap_m_transaccional
#     WHERE YEAR = 2026
#      AND MONTH = 6
#      AND DAY = 5
#      AND LOWER(TRIM(tipo_trx)) = "purchase"
#      AND LOWER(TRIM(estado_trx)) = "cleared"
#     GROUP BY 1,
#             2
#             )
# SELECT count(*)
# FROM outcome;
# """
# helper.obtener_dataframe(sql)

--------------------------------------------------------------------------------------------
    i      tipo                 nombre                  estado     hora_inicio   duracion   
--------------------------------------------------------------------------------------------
 473/473 DATAFRAME                                    descargando   11:49:10 PM             

2026-06-10 23:49:13 - [INFO] - 1 filas, 1 columnas, 00:02.0 consultando, 00:00.7 descargando, 00:00.0 convirtiendo


 473/473 DATAFRAME                                     finalizado   11:49:10 PM     00:02.9 
--------------------------------------------------------------------------------------------


,count(*)
0,71934


In [ ]:
# # Verificar cantidad de registros por partición
# # La última ingestión usada 2025-11-14 para obtener las transacciones. # MODIFICAR FECHA
# sql = """
# SELECT YEAR,
#        mes,
#        dia,
#        count(*) AS frec
# FROM proceso.mdo_adquirencia_trxs
# WHERE YEAR BETWEEN 2022 AND 2026
#   AND mes BETWEEN 1 AND 12
#   AND dia BETWEEN 1 AND 31
# GROUP BY 1,
#          2,
#          3
# ORDER BY YEAR DESC, mes DESC, dia DESC;
# """
# helper.obtener_dataframe(sql).head(20)

--------------------------------------------------------------------------------------------
    i      tipo                 nombre                  estado     hora_inicio   duracion   
--------------------------------------------------------------------------------------------
 474/474 DATAFRAME                                    descargando   11:49:15 PM             

2026-06-10 23:49:19 - [INFO] - 1,086 filas, 4 columnas, 00:02.7 consultando, 00:00.4 descargando, 00:00.0 convirtiendo


 474/474 DATAFRAME                                     finalizado   11:49:15 PM     00:03.4 
--------------------------------------------------------------------------------------------


,year,mes,dia,frec
0,2026,6,9,231252
1,2026,6,5,71934
2,2026,6,4,71104
3,2026,6,3,70762
4,2026,6,2,69175
5,2026,6,1,182108
6,2026,5,29,71893
7,2026,5,28,70476
8,2026,5,27,69595
9,2026,5,26,69417


## Construcción histórico transacciones por mes

In [ ]:

sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_trxs_mes_1 PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_adquirencia_trxs_mes_1 STORED AS PARQUET AS
SELECT cod_unico,
       cast(replace(left(cast(f_trx AS string), 7), '-', '') AS INT) AS periodo_trxs,
       count(*) AS num_trxs,
       sum(mnt_total_trxs) AS mnt_total_trxs
FROM proceso.mdo_adquirencia_trxs
WHERE YEAR BETWEEN 2022 AND """ + str(df_config.year.values[-1]) + """
AND MES BETWEEN 1 AND 12
GROUP BY 1,
       2;"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_trxs_mes_1;"""
helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
    i      tipo                    nombre                    estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 475/475      DROP proceso_vdm.mdo_adquirencia_trxs_mes_1   finalizado   11:49:24 PM     00:00.6 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
    i      tipo                    nombre                    estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 476/476    CREATE proceso_vdm.mdo_adquirencia_trxs_mes_1   finalizado   11:49:25 PM     00:06.9 
-------------------------------------------------------------------------------------------------
--------------------

## Construcción históricos

Se calcula la métrica uso para tres escenarios

- Todos los clientes
- Clientes nuevos
- Clientes viejos

Se dice que un cliente [todos, nuevos o viejos] tiene uso cuando realiza al menos una (1) transaccion en el año corriente. 

Importante tener en cuenta para clientes nuevos. Un cliente que vinculó adquirencia en el año 2024 [nuevo en 2024] y 
realizó un transacción en el mismo año suma a la métrica, en cambio si ese mismo cliente realiza una transacción en el año 2025 y 
siguientes no sumará a la métrica de nuevos.

In [ ]:
if actualizar == False:
    # Tabla que almacenará el número de vinculaciones por mes
    sql_drop = """DROP TABLE IF EXISTS proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist PURGE;"""
    helper.ejecutar_consulta(sql_drop)

    sql = """
    CREATE TABLE proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist  (
                    periodo DOUBLE,
                    num_vinc BIGINT,
                    tipo_cliente STRING
                    )
    STORED AS PARQUET
    TBLPROPERTIES ('transactional' = 'false');
    """
    helper.ejecutar_consulta(sql)

    sql_compute = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist;"""
    helper.ejecutar_consulta(sql_compute)

---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------------
 478/478      DROP ..._adquirencia_1_a_1_vinculaciones_hist   finalizado   11:49:39 PM     00:00.6 
---------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------------
 479/479    CREATE ..._adquirencia_1_a_1_vinculaciones_hist   finalizado   11:49:40 PM     00:00.6 
---------------------------------------------------------------------------------------------------


In [ ]:
if actualizar == False:
    # Tabla que almacenará comercios con trxs por periodo y tipo cliente
    sql_drop = """DROP TABLE IF EXISTS proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist PURGE;"""
    helper.ejecutar_consulta(sql_drop)

    sql = """
    CREATE TABLE proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist  (
                    codigo_unico VARCHAR,
                    periodo DOUBLE,
                    num_trxs BIGINT,
                    mnt_total_trxs DECIMAL(38,2),
                    tipo_cliente STRING
                    )
    STORED AS PARQUET
    TBLPROPERTIES ('transactional' = 'false');
    """
    helper.ejecutar_consulta(sql)

    sql_compute = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist;"""
    helper.ejecutar_consulta(sql_compute)

---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------------
 481/481      DROP ...cia_1_a_1_vinculaciones_con_trxs_hist   finalizado   11:49:44 PM     00:00.6 
---------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------------
 482/482    CREATE ...cia_1_a_1_vinculaciones_con_trxs_hist   finalizado   11:49:44 PM     00:00.6 
---------------------------------------------------------------------------------------------------


In [ ]:
# Construcción histórico métricas de uso de adquirencia
fecha_inicial_ts = pd.to_datetime(fecha_inicial_metricas)
fecha_final_ts = pd.to_datetime(fecha_final_metricas)
fechas = pd.date_range(start=fecha_inicial_ts, end=fecha_final_ts, freq='M')

# # Ir por las particiones del siguiente mes que contendrá las compras de los últimos días del reango de tiempo deseado
# pri_dia_part = fechas[-1] + relativedelta(days=1)
# pri_dia_part = pri_dia_part.date().isoformat() # Primer día de partición para buscar compras del primer día.
# ult_dia_part = fechas[-1] + relativedelta(days=10)
# ult_dia_part = ult_dia_part.date().isoformat()
# fechas_faltantes = pd.date_range(start=pri_dia_part, end=ult_dia_part, freq='D')

# # siguiente_dias = dt.datetime(siguientes_dias.year, siguientes_dias.month, calendar.monthrange(siguientes_dias.year, siguientes_dias.month)[1])
# fechas = fechas.append(fechas_faltantes)

# df_config
df_config = pd.DataFrame({'fechas': fechas})
df_config['year'] = df_config['fechas'].dt.year
df_config['month'] = df_config['fechas'].dt.month
df_config['day'] = df_config['fechas'].dt.day
df_config['periodo'] = df_config['year'] * 100 + df_config['month']
df_config['periodo_base_year'] = df_config['year'] * 100 + 1
df_config['periodo_viejos'] = df_config['fechas'].apply(lambda x: str(x.year - 1) + '12')
df_config['periodo_sgte_mes'] = df_config['fechas'].apply(lambda x: x + relativedelta(months=1))
df_config['year_sgte_mes'] = df_config['periodo_sgte_mes'].dt.year
df_config['month_sgte_mes'] = df_config['periodo_sgte_mes'].dt.month
df_config['periodo_sgte_mes'] = df_config['periodo_sgte_mes'].apply(lambda x: x.year * 100 + x.month)

# df_config['ult_6_meses_fin'] = df_config['fechas'].apply(lambda x: x - relativedelta(months=5))
# df_config['ult_6_meses_inicio'] = df_config['ult_6_meses_fin'].apply(lambda x: x.replace(day=1))
# df_config['year_ult_6_meses'] = df_config['ult_6_meses_fin'].dt.year
# df_config['month_ult_6_meses'] = df_config['ult_6_meses_fin'].dt.month
# df_config['periodo_ult_6_meses'] = df_config['year_ult_6_meses'] * 100 + df_config['month_ult_6_meses']
df_config

,fechas,year,month,day,periodo,periodo_base_year,periodo_viejos,periodo_sgte_mes,year_sgte_mes,month_sgte_mes
0,2022-01-31,2022,1,31,202201,202201,202112,202202,2022,2
1,2022-02-28,2022,2,28,202202,202201,202112,202203,2022,3
2,2022-03-31,2022,3,31,202203,202201,202112,202204,2022,4
3,2022-04-30,2022,4,30,202204,202201,202112,202205,2022,5
4,2022-05-31,2022,5,31,202205,202201,202112,202206,2022,6
5,2022-06-30,2022,6,30,202206,202201,202112,202207,2022,7
6,2022-07-31,2022,7,31,202207,202201,202112,202208,2022,8
7,2022-08-31,2022,8,31,202208,202201,202112,202209,2022,9
8,2022-09-30,2022,9,30,202209,202201,202112,202210,2022,10
9,2022-10-31,2022,10,31,202210,202201,202112,202211,2022,11


In [51]:
# Adicionar primer día de ingestión a df_config
df_config = df_config.merge(df_primer_dia_ing_gsap_m_comercios[['year_sgte_mes', 'month_sgte_mes', 'primer_dia_ing_gsap_m_comercios']],
                             on=['year_sgte_mes', 'month_sgte_mes'], how='left')
df_config

,fechas,year,month,day,periodo,periodo_base_year,periodo_viejos,periodo_sgte_mes,year_sgte_mes,month_sgte_mes,primer_dia_ing_gsap_m_comercios
0,2022-01-31,2022,1,31,202201,202201,202112,202202,2022,2,1
1,2022-02-28,2022,2,28,202202,202201,202112,202203,2022,3,1
2,2022-03-31,2022,3,31,202203,202201,202112,202204,2022,4,1
3,2022-04-30,2022,4,30,202204,202201,202112,202205,2022,5,2
4,2022-05-31,2022,5,31,202205,202201,202112,202206,2022,6,1
5,2022-06-30,2022,6,30,202206,202201,202112,202207,2022,7,1
6,2022-07-31,2022,7,31,202207,202201,202112,202208,2022,8,1
7,2022-08-31,2022,8,31,202208,202201,202112,202209,2022,9,1
8,2022-09-30,2022,9,30,202209,202201,202112,202210,2022,10,3
9,2022-10-31,2022,10,31,202210,202201,202112,202211,2022,11,1


In [52]:
df_config.head(20)

,fechas,year,month,day,periodo,periodo_base_year,periodo_viejos,periodo_sgte_mes,year_sgte_mes,month_sgte_mes,primer_dia_ing_gsap_m_comercios
0,2022-01-31,2022,1,31,202201,202201,202112,202202,2022,2,1
1,2022-02-28,2022,2,28,202202,202201,202112,202203,2022,3,1
2,2022-03-31,2022,3,31,202203,202201,202112,202204,2022,4,1
3,2022-04-30,2022,4,30,202204,202201,202112,202205,2022,5,2
4,2022-05-31,2022,5,31,202205,202201,202112,202206,2022,6,1
5,2022-06-30,2022,6,30,202206,202201,202112,202207,2022,7,1
6,2022-07-31,2022,7,31,202207,202201,202112,202208,2022,8,1
7,2022-08-31,2022,8,31,202208,202201,202112,202209,2022,9,1
8,2022-09-30,2022,9,30,202209,202201,202112,202210,2022,10,3
9,2022-10-31,2022,10,31,202210,202201,202112,202211,2022,11,1


In [ ]:
for row in df_config.itertuples():
    print('#' * 50)
    print('')
    print('Obteniendo Métrica 5X')
    print('')
    print('Mes de Análisis: ', str(row.year), '-', str(row.month))
    print('')

    print('Viejos')
    print('') 
    print('Obteniendo vinculaciones acumuladas al último mes del año anterior correspondiente al Mes de análisis')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_vinculaciones_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_vinculaciones_temp STORED AS PARQUET AS
      SELECT id_comercio as codigo_unico,
             """ + str(row.periodo) + """ AS periodo
      FROM resultados_vspc_medios_de_pago.gsap_m_comercios
      WHERE YEAR = """ + str(row.year_sgte_mes) + """
      AND MONTH = """ + str(row.month_sgte_mes) + """
      AND DAY = """ + str(row.primer_dia_ing_gsap_m_comercios) + """
      AND UPPER(estado_comercio) LIKE 'ACTIV%'
      AND CASE
               WHEN cast(id_comercio_padre AS BIGINT) = cast(nit AS BIGINT) THEN 1
               ELSE 0
            END = 1
      AND year(f_vinculacion_opy)*100 + month(f_vinculacion_opy) <= """ + row.periodo_viejos + """;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_vinculaciones_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando número de vinculados viejos en proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist')
    print('')
    sql = """
    INSERT INTO proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist
    SELECT periodo,
           count(*) AS num_vinc,
           'viejos' as tipo_cliente
    FROM proceso.mdo_adquirencia_vinculaciones_temp
    GROUP BY 1;
    """
    helper.ejecutar_consulta(sql)

    print('')
    print('Obtener las transacciones año base corrido')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_trxs_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_trxs_temp STORED AS PARQUET AS
    select cod_unico,
         sum(num_trxs) as num_trxs,
         sum(mnt_total_trxs) as mnt_total_trxs,
         'viejos' as tipo_cliente
    FROM proceso.mdo_adquirencia_trxs_mes_1
    WHERE periodo_trxs BETWEEN """ + str(row.periodo_base_year) + """ AND """ + str(row.periodo) + """
    GROUP BY 1;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_trxs_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando en tabla proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist')
    print('')
    
    sql = """
       INSERT INTO proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist
       SELECT a.codigo_unico,
           a.periodo,
           b.num_trxs,
           b.mnt_total_trxs,
           b.tipo_cliente
    FROM proceso.mdo_adquirencia_vinculaciones_temp AS a
    INNER JOIN proceso.mdo_adquirencia_trxs_temp AS b ON a.codigo_unico = b.cod_unico;
    """
    helper.ejecutar_consulta(sql)
    print('')


    print('Nuevos')

    print('') 
    print('Obteniendo vinculaciones correspondiente al Mes de análisis')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_vinculaciones_temp_1_new PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_vinculaciones_temp_1_new STORED AS PARQUET AS
      SELECT id_comercio as codigo_unico,
             """ + str(row.periodo) + """ AS periodo
      FROM resultados_vspc_medios_de_pago.gsap_m_comercios
      WHERE YEAR = """ + str(row.year_sgte_mes) + """
      AND MONTH = """ + str(row.month_sgte_mes) + """
      AND DAY = """ + str(row.primer_dia_ing_gsap_m_comercios) + """
      AND UPPER(estado_comercio) LIKE 'ACTIV%'
      AND CASE
               WHEN cast(id_comercio_padre AS BIGINT) = cast(nit AS BIGINT) THEN 1
               ELSE 0
            END = 1
      AND year(f_vinculacion_opy)*100 + month(f_vinculacion_opy) = """ + str(row.periodo) + """;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_vinculaciones_temp_1_new;"""
    helper.ejecutar_consulta(sql_compute)
    
    print('')
    print('Eliminar los nuevos que ya aparecen en los viejos, se puede presentar múltiples razones, entre ellas: ' \
    'adición de franquicias, la adición de una nueva franquicia genera un nuevo registro en la tabla')
    print('')
    print('Crear tabla con los viejos')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_vinculaciones_viejos_temp_new PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_vinculaciones_viejos_temp_new STORED AS PARQUET AS
    SELECT codigo_unico, periodo
    FROM proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist
    WHERE tipo_cliente = 'viejos'"""
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_vinculaciones_viejos_temp_new;"""
    helper.ejecutar_consulta(sql_compute)
    print('')

    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_vinculaciones_temp_new PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_vinculaciones_temp_new STORED AS PARQUET AS
    SELECT a.codigo_unico, a.periodo
    FROM proceso.mdo_adquirencia_vinculaciones_temp_1_new AS a
    LEFT ANTI JOIN proceso.mdo_adquirencia_vinculaciones_viejos_temp_new AS b on a.codigo_unico = b.codigo_unico
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_vinculaciones_temp_new;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando número de vinculados nuevos en proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist')
    print('')
    sql = """
    INSERT INTO proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist
    SELECT periodo,
           count(*) AS num_vinc,
           'nuevos' as tipo_cliente
    FROM proceso.mdo_adquirencia_vinculaciones_temp_new
    GROUP BY 1;
    """
    helper.ejecutar_consulta(sql)

    print('') 
    print('Obteniendo vinculaciones del año correspondiente, acumuladas hasta el Mes de análisis')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_vinculaciones_temp_1 PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_vinculaciones_temp_1 STORED AS PARQUET AS
      SELECT id_comercio as codigo_unico,
             """ + str(row.periodo) + """ AS periodo
      FROM resultados_vspc_medios_de_pago.gsap_m_comercios
      WHERE YEAR = """ + str(row.year_sgte_mes) + """
      AND MONTH = """ + str(row.month_sgte_mes) + """
      AND DAY = """ + str(row.primer_dia_ing_gsap_m_comercios) + """
      AND UPPER(estado_comercio) LIKE 'ACTIV%'
      AND CASE
               WHEN cast(id_comercio_padre AS BIGINT) = cast(nit AS BIGINT) THEN 1
               ELSE 0
            END = 1
      AND year(f_vinculacion_opy)*100 + month(f_vinculacion_opy) BETWEEN """ + str(row.periodo_base_year) + """ AND """ + str(row.periodo) + """;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_vinculaciones_temp_1;"""
    helper.ejecutar_consulta(sql_compute)
    
    print('')
    print('Eliminar los nuevos que ya aparecen en los viejos, se puede presentar múltiples razones, entre ellas: ' \
    'adición de franquicias, la adición de una nueva franquicia genera un nuevo registro en la tabla')
    print('')
    print('Crear tabla con los viejos')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_vinculaciones_viejos_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_vinculaciones_viejos_temp STORED AS PARQUET AS
    SELECT codigo_unico, periodo
    FROM proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist
    WHERE tipo_cliente = 'viejos'"""
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_vinculaciones_viejos_temp;"""
    helper.ejecutar_consulta(sql_compute)
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_vinculaciones_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_vinculaciones_temp STORED AS PARQUET AS
    SELECT a.codigo_unico, a.periodo
    FROM proceso.mdo_adquirencia_vinculaciones_temp_1 AS a
    LEFT ANTI JOIN proceso.mdo_adquirencia_vinculaciones_viejos_temp AS b on a.codigo_unico = b.codigo_unico
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_vinculaciones_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Obtener las transacciones año base corrido')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_trxs_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_trxs_temp STORED AS PARQUET AS
    select cod_unico,
         sum(num_trxs) as num_trxs,
         sum(mnt_total_trxs) as mnt_total_trxs,
         'nuevos' as tipo_cliente
    FROM proceso.mdo_adquirencia_trxs_mes_1
    WHERE periodo_trxs BETWEEN """ + str(row.periodo_base_year) + """ AND """ + str(row.periodo) + """
    GROUP BY 1;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_trxs_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando en tabla proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist')
    print('')
    
    sql = """
    INSERT INTO proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist
    SELECT a.codigo_unico,
           a.periodo,
           b.num_trxs,
           b.mnt_total_trxs,
           b.tipo_cliente
    FROM proceso.mdo_adquirencia_vinculaciones_temp AS a
    INNER JOIN proceso.mdo_adquirencia_trxs_temp AS b ON a.codigo_unico = b.cod_unico;
    """
    helper.ejecutar_consulta(sql)
    print('')

    print('Todos')
    print('') 
    print('Obteniendo vinculaciones acumuladas hasta el Mes de análisis')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_vinculaciones_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_vinculaciones_temp STORED AS PARQUET AS
      SELECT id_comercio as codigo_unico,
             """ + str(row.periodo) + """ AS periodo
      FROM resultados_vspc_medios_de_pago.gsap_m_comercios
      WHERE YEAR = """ + str(row.year_sgte_mes) + """
      AND MONTH = """ + str(row.month_sgte_mes) + """
      AND DAY = """ + str(row.primer_dia_ing_gsap_m_comercios) + """
      AND UPPER(estado_comercio) LIKE 'ACTIV%'
      AND CASE
               WHEN cast(id_comercio_padre AS BIGINT) = cast(nit AS BIGINT) THEN 1
               ELSE 0
            END = 1
      AND year(f_vinculacion_opy)*100 + month(f_vinculacion_opy) <= """ + str(row.periodo) + """;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_vinculaciones_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando número de vinculados (todos) en proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist')
    print('')
    sql = """
    INSERT INTO proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist
    SELECT periodo,
           count(*) AS num_vinc,
           'todos' as tipo_cliente
    FROM proceso.mdo_adquirencia_vinculaciones_temp
    GROUP BY 1;
    """
    helper.ejecutar_consulta(sql)

    print('')
    print('Obtener las transacciones año base corrido')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_trxs_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_trxs_temp STORED AS PARQUET AS
    select cod_unico,
         sum(num_trxs) as num_trxs,
         sum(mnt_total_trxs) as mnt_total_trxs,
         'todos' as tipo_cliente
    FROM proceso.mdo_adquirencia_trxs_mes_1
    WHERE periodo_trxs BETWEEN """ + str(row.periodo_base_year) + """ AND """ + str(row.periodo) + """
    GROUP BY 1;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_trxs_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando en tabla proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist')
    print('')
    
    sql = """
    INSERT INTO proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist
    SELECT a.codigo_unico,
           a.periodo,
           b.num_trxs,
           b.mnt_total_trxs,
           b.tipo_cliente
    FROM proceso.mdo_adquirencia_vinculaciones_temp AS a
    INNER JOIN proceso.mdo_adquirencia_trxs_temp AS b ON a.codigo_unico = b.cod_unico;
    """
    helper.ejecutar_consulta(sql)
    print('')

    

##################################################

Obteniendo Métrica 5X

Mes de Análisis:  2022 - 1

Viejos

Obteniendo vinculaciones acumuladas al último mes del año anterior correspondiente al Mes de análisis

---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------------
 484/484      DROP ...so.mdo_adquirencia_vinculaciones_temp   finalizado   11:55:11 PM     00:00.9 
---------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
--------------------------------------------------------------------------------------

# Evitar ciclo de vida zona proceso_vdm

In [ ]:
# Tabla proceso.mdo_adquirencia_trxs'

sql_drop_1 = f"""DROP TABLE IF EXISTS proceso.mdo_adquirencia_trxs PURGE;"""
helper.ejecutar_consulta(sql_drop_1)

sql_proceso = f"""CREATE TABLE proceso.mdo_adquirencia_trxs STORED AS PARQUET AS
    SELECT cod_unico,
           f_trx,
           num_trxs,
           mnt_total_trxs,
           DIA
    FROM proceso.mdo_adquirencia_trxs;"""
helper.ejecutar_consulta(sql_proceso)

sql_compute_1 = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_trxs;"""
helper.ejecutar_consulta(sql_compute_1)

sql_drop_2 = f"""DROP TABLE IF EXISTS proceso.mdo_adquirencia_trxs PURGE;"""
helper.ejecutar_consulta(sql_drop_2)

sql_vdm = f"""CREATE TABLE proceso.mdo_adquirencia_trxs STORED AS PARQUET AS
    SELECT cod_unico,
           f_trx,
           num_trxs,
           mnt_total_trxs,
           DIA
    FROM proceso.mdo_adquirencia_trxs;"""
helper.ejecutar_consulta(sql_vdm)

sql_compute_2 = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_trxs;"""
helper.ejecutar_consulta(sql_compute_2)

sql_drop_3 = f"""DROP TABLE IF EXISTS proceso.mdo_adquirencia_trxs PURGE;"""

--------------------------------------------------------------------------------
  i   tipo             nombre               estado     hora_inicio   duracion   
--------------------------------------------------------------------------------
 3/3   DROP proceso.mdo_adquirencia_trxs   finalizado   10:08:06 AM     00:00.6 
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
  i   tipo             nombre               estado     hora_inicio   duracion   
--------------------------------------------------------------------------------
 4/4 CREATE proceso.mdo_adquirencia_trxs   finalizado   10:08:07 AM     00:04.5 
--------------------------------------------------------------------------------
---------------------------------------------------------------------------------
  i   tipo              nombre               estado     hora_inicio   duracion   
--------------------------

In [5]:
# Tabla proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist'

sql_drop_1 = f"""DROP TABLE IF EXISTS proceso.mdo_adquirencia_1_a_1_vinculaciones_hist PURGE;"""
helper.ejecutar_consulta(sql_drop_1)

sql_proceso = f"""CREATE TABLE proceso.mdo_adquirencia_1_a_1_vinculaciones_hist STORED AS PARQUET AS
    SELECT periodo,
           num_vinc,
           tipo_cliente
    FROM proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist;"""
helper.ejecutar_consulta(sql_proceso)

sql_compute_1 = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_1_a_1_vinculaciones_hist;"""
helper.ejecutar_consulta(sql_compute_1)

sql_drop_2 = f"""DROP TABLE IF EXISTS proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist PURGE;"""
helper.ejecutar_consulta(sql_drop_2)

sql_vdm = f"""CREATE TABLE proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist STORED AS PARQUET AS
    SELECT periodo,
           num_vinc,
           tipo_cliente
    FROM proceso.mdo_adquirencia_1_a_1_vinculaciones_hist;"""
helper.ejecutar_consulta(sql_vdm)

sql_compute_2 = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist;"""
helper.ejecutar_consulta(sql_compute_2)

sql_drop_3 = f"""DROP TABLE IF EXISTS proceso.mdo_adquirencia_1_a_1_vinculaciones_hist PURGE;"""

---------------------------------------------------------------------------------------------
  i   tipo                    nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 9/9    DROP ..._adquirencia_1_a_1_vinculaciones_hist   finalizado   10:08:19 AM     00:00.6 
---------------------------------------------------------------------------------------------
-----------------------------------------------------------------------------------------------
   i    tipo                    nombre                     estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------------------
 10/10  CREATE ..._adquirencia_1_a_1_vinculaciones_hist   finalizado   10:08:20 AM     00:01.3 
-----------------------------------------------------------------------------------------------
--------------------------------------------------

In [6]:
# Tabla proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist

sql_drop_1 = f"""DROP TABLE IF EXISTS proceso.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist PURGE;"""
helper.ejecutar_consulta(sql_drop_1)

sql_proceso = f"""CREATE TABLE proceso.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist STORED AS PARQUET AS
    SELECT codigo_unico,
           periodo,
           num_trxs,
           mnt_total_trxs,
           tipo_cliente
    FROM proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist;"""
helper.ejecutar_consulta(sql_proceso)

sql_compute_1 = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist;"""
helper.ejecutar_consulta(sql_compute_1)

sql_drop_2 = f"""DROP TABLE IF EXISTS proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist PURGE;"""
helper.ejecutar_consulta(sql_drop_2)

sql_vdm = f"""CREATE TABLE proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist STORED AS PARQUET AS
    SELECT codigo_unico,
           periodo,
           num_trxs,
           mnt_total_trxs,
           tipo_cliente
    FROM proceso.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist;"""
helper.ejecutar_consulta(sql_vdm)

sql_compute_2 = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist;"""
helper.ejecutar_consulta(sql_compute_2)

sql_drop_3 = f"""DROP TABLE IF EXISTS proceso.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist PURGE;"""
helper.ejecutar_consulta(sql_drop_3)

-----------------------------------------------------------------------------------------------
   i    tipo                    nombre                     estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------------------
 15/15    DROP ...cia_1_a_1_vinculaciones_con_trxs_hist   finalizado   10:08:26 AM     00:00.6 
-----------------------------------------------------------------------------------------------
-----------------------------------------------------------------------------------------------
   i    tipo                    nombre                     estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------------------
 16/16  CREATE ...cia_1_a_1_vinculaciones_con_trxs_hist   finalizado   10:08:26 AM     00:02.0 
-----------------------------------------------------------------------------------------------
----------------------------------------